##### Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

##### Loading of dataset

In [ ]:
import os

print(os.getcwd())

In [ ]:
df = pd.read_csv("../data/drugs_with_barcodes.csv")

In [ ]:
# Display the first and the last 5 rows
df.head()

In [ ]:
df.tail()

In [ ]:
# Dataset Shape
df.shape

In [ ]:
# View Columns name
df.columns

In [ ]:
# Check for Data Type
df.info()

In [ ]:
# Statistical Summary of the Dataset
df.describe()

##### Checking for Missing Values

In [ ]:
df.isnull().sum()

##### Checking for Duplicate

In [ ]:
df.duplicated().sum()

In [ ]:
# Correct the duplicate
df.drop_duplicates(inplace=True)

#### Check for Distribution

In [ ]:
df["verification_status"].value_counts()

In [ ]:
df["verification_status"].value_counts().plot(kind="bar")
plt.title("Verification Status Distribution")
plt.xlabel("Status")
plt.ylabel("Count")
plt.show()

##### Manufacturer Distribution

In [ ]:
df["manufacturer"].value_counts().head(10)

In [ ]:
df["manufacturer"].value_counts() .head(10).plot(kind="bar")
plt.title("Top 10 Manufacturers")
plt.xticks(rotation=90)
plt.show()

##### Product Type Distribution

In [ ]:
df["product_type"].value_counts()

In [ ]:
df["product_type"].value_counts().plot(kind="bar")
plt.title("product Type")
plt.xticks(rotation=90)
plt.show()

##### Route Distribution

In [ ]:
df["route"].value_counts().head(10)

In [ ]:
df["route"].value_counts().head(10).plot(kind="bar")
plt.title("First 10 Routes")
plt.xticks(rotation=90)
plt.show()

##### Unique Value Check

In [ ]:
df.nunique()

In [ ]:
# Checking Barcode Uniqueness
df["barcode"].is_unique

In [ ]:
df["verification_status"].is_unique

#### OBSERVATIONS

* The dataset contains 137,468 records and 15 features (columns).
* The dataset consists of one (1) floating-point column, two (2) integer columns, and twelve (12) string (object) columns.
* The dataset contains missing values, with the route column having the highest number of missing entries.
* No duplicate records were found in the dataset.
* The target variable, verification_status, contains two classes:
    * Not Fake – approximately 80% of the records.
    * Fake – approximately 20% of the records.
This indicates that the dataset is imbalanced, which will be considered during model training and evaluation.

* The dataset contains products from multiple manufacturers, with some manufacturers contributing significantly more records than others.
* The barcode, product_id, and batch_number columns contain unique values for every record, making them suitable as unique identifiers

### Data Preprocessing

In [ ]:
# Checking Missing Value again
df.isnull().sum()

##### Handle Missing values

In [ ]:
columns_to_fill = [
    "route",
    "brand_name",
    "listing_expiration_date",
    "generic_name"
]

df[columns_to_fill] = df[columns_to_fill].fillna("Unknown")

In [ ]:
df.isnull().sum()

##### Convert date columns to datetime format

In [ ]:
date_columns = [
    "marketing_start_date",
    "listing_expiration_date",
    "manufacture_date",
    "expiry_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

##### Feature Engineering

In [ ]:
# Creating New Features

# a.  Extracting Meaningful information from the date, like the Year
df["marketing_year"] = df["marketing_start_date"].dt.year
df["listing_expiry_year"] = df["listing_expiration_date"].dt.year
df["manufacture_year"] = df["manufacture_date"].dt.year
df["expiry_year"] = df["expiry_date"].dt.year

In [ ]:
# b. Drug Shelf Life

df["shelf_life_days"] = (
    df["expiry_date"] - df["manufacture_date"]
).dt.days

##### Encode the Target

In [ ]:
df["verification_status"] = df["verification_status"].map({
    "Not Fake": 0,
    "Fake": 1
})

In [ ]:
# Create a Copy of the Dataset
df_ml = df.copy()

##### Feature Selection

In [ ]:
features = [
    "generic_name",
    "manufacturer",
    "brand_name",
    "dosage_form",
    "product_type",
    "route",
    "marketing_year",
    "listing_expiry_year",
    "manufacture_year",
    "expiry_year",
    "shelf_life_days"
]

X = df_ml[features]

y = df_ml["verification_status"]

In [ ]:
# Seperating the Categorical Features
# a. High Cardinality Features: Those with thousands of unique values 

high_cardinality = [
    "generic_name",
    "manufacturer",
    "brand_name"
]

In [ ]:
# b. Low Cardinality Features: Those with relatively few unique values.

low_cardinality = [
    "dosage_form",
    "product_type",
    "route"
]

##### Encode Features for Categorical Columns


In [ ]:
# Using Frequency Encoding (High Cardinality) for Categorical Variables

for col in high_cardinality:
    frequency = X[col].value_counts()

    X[col + "_freq"] = X[col].map(frequency)

In [ ]:
# Checking the new Columns
X[[
    "generic_name_freq",
    "manufacturer_freq",
    "brand_name_freq"
]].head()

In [ ]:
# Removing the original High cardinality columns
X = X.drop(columns=high_cardinality)

In [ ]:
# One Hot Encoding for the Low Cardinality  Features
X = pd.get_dummies(
    X,
    columns=low_cardinality,
    drop_first=True,
    dtype=int
)

In [ ]:
print(X.shape)

In [ ]:
X.info()

In [ ]:
X.isnull().sum().sum()

In [ ]:
X.isnull().sum().sort_values(ascending=False)

In [ ]:
# Handling Missing Value after Encoding with Median
X["listing_expiry_year"] = X["listing_expiry_year"].fillna(
    X["listing_expiry_year"].median()
)

In [ ]:
print(X.isnull().sum().sum())

##### Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# Verify the split

print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels  :", y_test.shape)

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

##### Saving the Final dataset i used for Preprocessing

In [ ]:
# Combine features and target
processed_data = X.copy()
processed_data["verification_status"] = y

# Save to CSV
processed_data.to_csv(
    "../data/processed_barcode_data.csv",
    index=False
)

print("Processed dataset saved successfully!")
print(processed_data.shape)